<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Explain an online shop's sales

**[C04-M07-L01 · Explain an online shop's sales](https://learning.nextia-ai.com/courses/sql/m07/explain-an-online-shops-sales/)** · C04, SQL and Data Preparation for AI · Module 7, case study 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** load a real online shop's export of 1,067,371 invoice lines into SQLite, find its traps, and build four tables with keys. You reconcile the revenue, explain the monthly sales with `LAG`, find the orders behind the return spikes, measure whether new customers come back with CTEs, and rank products per country with ties. You deliver a checked monthly reporting table with a lineage note.

| | |
|---|---|
| **Time** | About 150 minutes with the lesson page; the notebook runs in about one minute |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6. |
| **You should know** | Keys and orphans, from [Relational concepts](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/). Joins that add rows, from [Join tables](https://learning.nextia-ai.com/courses/sql/m02/join-tables/). CTEs, from [Readable queries](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/). Window functions, from [Time and ranking](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/). Validation rules, from [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/). |
| **You do not need** | The course project. This notebook downloads its own data and uses all of it (no sample). |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4) with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m07/explain-an-online-shops-sales/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M07-L01-explain-an-online-shops-sales/explain-an-online-shops-sales-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the data

Run the next cell. It makes a folder `explain-an-online-shops-sales` with a `data` folder in it. It downloads `online_retail_ii.csv.gz` (15 MB) from the course's labs repository and checks the file's checksum (SHA-256). Everything this notebook writes goes into `explain-an-online-shops-sales`. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: data/online_retail_ii.csv.gz (15.0 MB)`. If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `explain-an-online-shops-sales` (in Colab: the **Files** panel on the left), then run the cell again.

The data is **Online Retail II** by Daqing Chen, from the UCI Machine Learning Repository ([doi.org/10.24432/C5CG6D](https://doi.org/10.24432/C5CG6D)), under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) licence. The course joined the two Excel sheets of the source into one compressed CSV file and changed nothing else; the [dataset card](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/case-studies/data/online_retail_ii.md) lists the details. It is real data of a UK online shop. Customer IDs are numbers only; do not try to identify anyone.

In [ ]:
# Setup: download the shop's export into explain-an-online-shops-sales/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/"
FILES = {"online_retail_ii.csv.gz": "c5ab45f73700a1411440c2cd6b7d85611d22cdc72b3594c2d29d37261e977e4f"}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("explain-an-online-shops-sales").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder explain-an-online-shops-sales and run again.")

os.chdir(WORK)  # work in the case-study folder
DATA_SHA256 = FILES["online_retail_ii.csv.gz"]
print("Ready:", ", ".join(f"data/{n} ({Path('data', n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A UK online shop sells gifts and homeware, often to small shops that resell them. It is December 2011. **Ines**, the merchandising lead, must order next Christmas's stock from suppliers in the spring. **Kwame**, a finance analyst, has three problems: three reports give three different revenue totals, the board wants a stricter returns policy after a January with 13.6% returns, and a dashboard says that new customers come back less and less. Ines and Kwame are made up; the data is real.

Your job: a small, checked database of the shop's sales, a monthly reporting table that finance can trust, and an answer to each question. The [lesson page](https://learning.nextia-ai.com/courses/sql/m07/explain-an-online-shops-sales/#the-situation) tells the full story.

## 2. The data

The export is one table of **invoice lines**: one product on one invoice. Run the next cell. It reads the file with pandas, adds `line_no` (the line's position in the file), and writes everything, unchanged, into a new SQLite database `shop.db`, in the table `raw_lines`. It takes about 5 seconds.

> **Check.** You should see `raw_lines: 1,067,371 lines, 9 columns`.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

raw = pd.read_csv("data/online_retail_ii.csv.gz",
                  dtype={"Invoice": str, "StockCode": str, "Customer ID": "Int64"})
raw.columns = ["invoice", "stock_code", "description", "quantity",
               "invoice_date", "price", "customer_id", "country"]
raw.insert(0, "line_no", range(1, len(raw) + 1))  # the line's position in the file

Path("shop.db").unlink(missing_ok=True)  # start from an empty database
con = sqlite3.connect("shop.db")
raw.to_sql("raw_lines", con, index=False)
print(f"raw_lines: {len(raw):,} lines, {raw.shape[1]} columns")

**What this tells us.** The raw table is a copy of the file, plus one column: `line_no`. Keep a raw copy that nobody changes, so that you can trace every later number back to it.

Run the next two cells: the size of the export, and its first four lines.

In [ ]:
sql("""
SELECT COUNT(*)                   AS lines,
       COUNT(DISTINCT invoice)    AS invoices,
       COUNT(DISTINCT stock_code) AS stock_codes,
       COUNT(DISTINCT customer_id) AS customers,
       MIN(invoice_date)          AS first_line,
       MAX(invoice_date)          AS last_line
FROM raw_lines
""")

In [ ]:
sql("""
SELECT line_no, invoice, stock_code, description, quantity,
       invoice_date, price, customer_id, country
FROM raw_lines
WHERE line_no <= 4
""")

> **Check.** You should see 53,628 invoices, 5,942 customers, and dates from 2009-12-01 to 2011-12-09.

**What this tells us.** Two years and nine days of sales. The first four lines are one invoice, `489434`: one customer bought four products at 07:45. One row is one line of an invoice; that is the **grain** of the export.

## 3. First checks: find the traps

> **Predict.** What share of the lines has no customer ID: under 5%, about 10% or more than 20%?

Run the next cell. It counts the lines with no customer, the returns (invoice starts with `C`), the adjustments (`A`) and the lines with a price of 0 or below.

In [ ]:
sql("""
SELECT COUNT(*)                                          AS lines,
       SUM(customer_id IS NULL)                          AS no_customer,
       ROUND(100.0 * SUM(customer_id IS NULL) / COUNT(*), 1) AS pct_no_customer,
       SUM(invoice LIKE 'C%')                            AS return_lines,
       SUM(invoice LIKE 'A%')                            AS adjustment_lines,
       SUM(price <= 0)                                   AS price_0_or_below
FROM raw_lines
""")

> **Check.** You should see 243,007 lines without a customer (22.8%) and 19,494 return lines.

**What this tells us.** A line without a customer is still a sale: a guest bought something. So these lines count in revenue, but not in anything about customers.

Run the next cell. It finds lines that are identical in every column.

In [ ]:
sql("""
WITH repeated AS (
    SELECT invoice_date, COUNT(*) AS copies
    FROM raw_lines
    GROUP BY invoice, stock_code, description, quantity,
             invoice_date, price, customer_id, country
    HAVING COUNT(*) > 1
)
SELECT COUNT(*)        AS groups,
       SUM(copies - 1) AS extra_lines,
       SUM(CASE WHEN invoice_date >= '2010-12-01' AND invoice_date < '2010-12-10'
                THEN copies - 1 ELSE 0 END) AS extra_lines_1_to_9_dec_2010
FROM repeated
""")

> **Check.** You should see 32,907 groups and 34,335 extra lines; 22,844 of them are from 1 to 9 December 2010.

**What this tells us.** Two thirds of the repeats are in nine days. That is not chance. The source was an Excel file with one sheet per year. If both sheets contain those nine days, the file holds them twice.

Run the next cell. `LAG` looks at the line before each line, in file order, and the query keeps the lines whose time goes **backwards**.

In [ ]:
sql("""
WITH in_file_order AS (
    SELECT line_no, invoice, invoice_date,
           LAG(invoice_date) OVER (ORDER BY line_no) AS previous_date
    FROM raw_lines
)
SELECT line_no, invoice, previous_date, invoice_date
FROM in_file_order
WHERE invoice_date < previous_date
""")

> **Check.** You should see exactly one row: line 525,462.

**What this tells us.** In the whole file, time goes backwards once: from 9 December 2010 to 1 December 2010. Line 525,462 is the first line of the second sheet. Run the next cell to compare the nine days in the two sheets.

In [ ]:
sql("""
WITH nine_days AS (
    SELECT *, CASE WHEN line_no < 525462 THEN 1 ELSE 2 END AS sheet
    FROM raw_lines
    WHERE invoice_date >= '2010-12-01' AND invoice_date < '2010-12-10'
),
per_line AS (
    SELECT SUM(sheet = 1) AS in_sheet_1, SUM(sheet = 2) AS in_sheet_2
    FROM nine_days
    GROUP BY invoice, stock_code, description, quantity,
             invoice_date, price, customer_id, country
)
SELECT (SELECT COUNT(*) FROM nine_days WHERE sheet = 1) AS lines_sheet_1,
       (SELECT COUNT(*) FROM nine_days WHERE sheet = 2) AS lines_sheet_2,
       COUNT(*)                                     AS different_lines,
       (SELECT ROUND(SUM(quantity * price)) FROM nine_days WHERE sheet = 2) AS amount_sheet_2
FROM per_line
WHERE in_sheet_1 <> in_sheet_2
""")

> **Check.** You should see 22,523 lines in each sheet and 0 different lines.

**What this tells us.** The second sheet repeats the nine days of the first sheet exactly. These 22,523 lines (£377,488) are a copy, so you remove them. Run the next cell to see the repeats that are left **inside** one sheet.

In [ ]:
sql("""
WITH one_copy AS (
    SELECT * FROM raw_lines
    WHERE NOT (line_no >= 525462 AND invoice_date < '2010-12-10')
),
repeated AS (
    SELECT COUNT(*) AS copies, quantity * price AS amount
    FROM one_copy
    GROUP BY invoice, stock_code, description, quantity,
             invoice_date, price, customer_id, country
    HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS groups, SUM(copies - 1) AS extra_lines,
       ROUND(SUM((copies - 1) * amount)) AS extra_amount,
       (SELECT ROUND(SUM(quantity * price)) FROM one_copy) AS all_lines_amount
FROM repeated
""")

> **Check.** You should see 11,812 extra lines worth £54,228.

**What this tells us.** These repeats are inside one sheet. They may be the same product typed twice on one invoice, which can be real. Nobody can prove that they are copies, and they are 0.3% of the amount. So you **keep** them and write the choice down.

Run the next three cells: the invoices, the product codes, and the names of one product.

In [ ]:
sql("""
WITH per_invoice AS (
    SELECT invoice,
           COUNT(DISTINCT customer_id)  AS customer_ids,
           COUNT(DISTINCT country)      AS countries,
           COUNT(DISTINCT invoice_date) AS times,
           MIN(invoice_date)            AS first_time,
           MAX(invoice_date)            AS last_time
    FROM raw_lines
    GROUP BY invoice
)
SELECT COUNT(*)                AS invoices,
       SUM(customer_ids > 1)   AS several_customers,
       SUM(countries > 1)      AS several_countries,
       SUM(times > 1)          AS several_times,
       MAX(ROUND((julianday(last_time) - julianday(first_time)) * 24 * 60)) AS max_minutes_apart,
       SUM(substr(first_time, 1, 7) <> substr(last_time, 1, 7)) AS cross_a_month
FROM per_invoice
""")

In [ ]:
sql("""
SELECT COUNT(DISTINCT stock_code)        AS codes,
       COUNT(DISTINCT UPPER(stock_code)) AS codes_ignoring_case,
       (SELECT COUNT(*) FROM (
            SELECT stock_code FROM raw_lines
            GROUP BY stock_code
            HAVING COUNT(DISTINCT description) > 1)) AS codes_with_several_descriptions
FROM raw_lines
""")

In [ ]:
sql("""
SELECT description, COUNT(*) AS lines,
       SUM(price > 0)           AS lines_with_a_price,
       SUM(customer_id IS NULL) AS lines_without_customer
FROM raw_lines
WHERE stock_code = '22423'
GROUP BY description
ORDER BY lines DESC
""")

> **Check.** You should see 83 invoices with several times (at most 9 minutes apart, none across a month), 5,305 codes but 5,132 when you ignore case, and 1,232 codes with several descriptions.

**What this tells us.** An invoice always has one customer and one country, so these can move to an `invoices` table. An invoice can have two times a few minutes apart; you keep the first. Codes such as `15056bl` and `15056BL` are one product. A product's "names" include stock notes like `damaged`, on lines with no price and no customer.

Run the next cell. It lists the codes that do not start with five digits.

In [ ]:
sql("""
SELECT stock_code, COUNT(*) AS lines, MIN(description) AS example,
       ROUND(SUM(quantity * price)) AS amount
FROM raw_lines
WHERE stock_code NOT GLOB '[0-9][0-9][0-9][0-9][0-9]*'
GROUP BY stock_code
ORDER BY lines DESC
""")

**What this tells us.** Most of these are not goods: postage (`POST`, `DOT`), manual corrections (`M`), discounts, bank charges, Amazon fees and gift vouchers. They are real money, but not sales of products. `DCGS0058` and `PADS` are real products. So the rule lists the non-goods codes by name.

## 4. Build four tables with keys

The decisions from the checks:

| Problem | Decision |
|---|---|
| Sheet 2 repeats 1 to 9 Dec 2010 | Remove the 22,523 copied lines (`line_no >= 525462` and before 10 Dec 2010). |
| Repeats inside one sheet | Keep; record them as a limit. |
| Codes in two cases | Use `UPPER(stock_code)`. |
| Several descriptions per code | Keep the name used most often. |
| Two times on one invoice | Keep the first time. |
| Postage, fees, adjustments | Keep in the tables, mark `is_goods = 0`, leave out of sales. |
| No customer ID | Keep: a guest sale. `customer_id` is NULL. |

Run the next cell. It makes a view of the lines without the copies, and four empty tables with primary and foreign keys.

In [ ]:
con.executescript("""
-- One copy of each line: sheet 2 repeats 1 to 9 December 2010 of sheet 1.
-- Codes are upper-cased: '15056bl' and '15056BL' are one product.
CREATE VIEW source_lines AS
SELECT line_no, invoice, UPPER(stock_code) AS stock_code, description,
       quantity, invoice_date, price, customer_id, country
FROM raw_lines
WHERE NOT (line_no >= 525462 AND invoice_date < '2010-12-10');

CREATE TABLE products (
    stock_code  TEXT PRIMARY KEY,
    description TEXT,
    is_goods    INTEGER NOT NULL CHECK (is_goods IN (0, 1))
);
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    country     TEXT NOT NULL
);
CREATE TABLE invoices (
    invoice     TEXT PRIMARY KEY,
    invoiced_at TEXT NOT NULL,
    customer_id INTEGER REFERENCES customers (customer_id),
    country     TEXT NOT NULL,
    kind        TEXT NOT NULL CHECK (kind IN ('sale', 'return', 'adjustment'))
);
CREATE TABLE invoice_lines (
    line_no    INTEGER PRIMARY KEY,
    invoice    TEXT NOT NULL REFERENCES invoices (invoice),
    stock_code TEXT NOT NULL REFERENCES products (stock_code),
    quantity   INTEGER NOT NULL,
    unit_price REAL NOT NULL
);
""")
print("Created the view source_lines and four empty tables.")

Run the next cell. It fills the tables. Each `INSERT` uses `ROW_NUMBER()` or `GROUP BY` to make exactly one row per key. It takes about 15 seconds.

> **Predict.** How many rows will `invoice_lines` have? Start from 1,067,371.

In [ ]:
con.executescript("""
-- products: the name used most often; 0 for postage, fees and adjustments
INSERT INTO products
SELECT stock_code, description,
       CASE WHEN stock_code IN ('POST', 'DOT', 'C2', 'C3', 'M', 'D', 'S', 'B', 'BANK CHARGES', 'ADJUST', 'ADJUST2', 'AMAZONFEE', 'CRUK', 'TEST001', 'TEST002')
              OR stock_code LIKE 'GIFT%' THEN 0 ELSE 1 END
FROM (
    SELECT stock_code, description,
           ROW_NUMBER() OVER (PARTITION BY stock_code
                              ORDER BY description IS NULL, COUNT(*) DESC, description) AS pick
    FROM source_lines
    GROUP BY stock_code, description
)
WHERE pick = 1;

-- customers: the country of the customer's latest invoice
INSERT INTO customers
SELECT customer_id, country
FROM (
    SELECT customer_id, country,
           ROW_NUMBER() OVER (PARTITION BY customer_id
                              ORDER BY invoice_date DESC, line_no DESC) AS pick
    FROM source_lines
    WHERE customer_id IS NOT NULL
)
WHERE pick = 1;

-- invoices: one row each; the first time if the lines have two
INSERT INTO invoices
SELECT invoice, MIN(invoice_date), MAX(customer_id), MAX(country),
       CASE WHEN invoice LIKE 'C%' THEN 'return'
            WHEN invoice LIKE 'A%' THEN 'adjustment'
            ELSE 'sale' END
FROM source_lines
GROUP BY invoice;

INSERT INTO invoice_lines
SELECT line_no, invoice, stock_code, quantity, price
FROM source_lines;

CREATE INDEX lines_by_invoice ON invoice_lines (invoice);
CREATE INDEX lines_by_product ON invoice_lines (stock_code);
CREATE INDEX invoices_by_customer ON invoices (customer_id);
""")
for table in ["raw_lines", "products", "customers", "invoices", "invoice_lines"]:
    print(f"{table:14} {con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]:>9,} rows")

> **Check.** You should see 5,132 products, 5,942 customers, 53,628 invoices and 1,044,848 invoice lines (1,067,371 − 22,523).

Run the next cell. Each check counts problems, so 0 is good.

In [ ]:
sql("""
SELECT 'lines without their invoice' AS check_name, COUNT(*) AS problems
FROM invoice_lines AS l LEFT JOIN invoices AS i ON i.invoice = l.invoice
WHERE i.invoice IS NULL
UNION ALL
SELECT 'lines without their product', COUNT(*)
FROM invoice_lines AS l LEFT JOIN products AS p ON p.stock_code = l.stock_code
WHERE p.stock_code IS NULL
UNION ALL
SELECT 'invoices with an unknown customer', COUNT(*)
FROM invoices AS i LEFT JOIN customers AS c ON c.customer_id = i.customer_id
WHERE i.customer_id IS NOT NULL AND c.customer_id IS NULL
UNION ALL
SELECT 'invoices without lines', COUNT(*)
FROM invoices AS i
WHERE NOT EXISTS (SELECT 1 FROM invoice_lines AS l WHERE l.invoice = i.invoice)
UNION ALL
SELECT 'raw lines not accounted for',
       (SELECT COUNT(*) FROM raw_lines) - (SELECT COUNT(*) FROM invoice_lines) - 22523
""")

> **Check.** You should see 0 in every row.

**What this tells us.** Every line has its invoice and its product, every customer ID exists, and every raw line is either in `invoice_lines` or one of the 22,523 copies. Run the next cell to see what kinds of lines the tables hold.

In [ ]:
sql("""
SELECT i.kind, p.is_goods, COUNT(*) AS lines,
       COUNT(DISTINCT i.invoice) AS invoices,
       SUM(i.customer_id IS NULL) AS lines_without_customer
FROM invoice_lines AS l
JOIN invoices AS i ON i.invoice = l.invoice
JOIN products AS p ON p.stock_code = l.stock_code
GROUP BY i.kind, p.is_goods
ORDER BY i.kind, p.is_goods DESC
""")

**What this tells us.** Almost all lines are sales of goods. Returns of goods are 17,974 lines.

> **Your turn.** How many **sale** invoices have no customer (guest orders)? Write a query on `invoices` that returns one row and one column, then run the check.

In [ ]:
guest_orders = """
-- Write your query here
"""
sql(guest_orders)

In [ ]:
check(guest_orders, rows=1, columns=1, expected="ac37db3a20bd0651")

## 5. Reconcile the revenue

Kwame's three reports differ. First, define the lines that count as sales of goods. Run the next cell. It makes the view `goods_lines`: goods only, a price above 0, sales and returns, with the invoice's facts on each line.

In [ ]:
con.executescript("""
-- The lines that count as sales of goods, with the facts of their invoice.
CREATE VIEW goods_lines AS
SELECT l.line_no, l.invoice, i.kind, i.invoiced_at,
       strftime('%Y-%m', i.invoiced_at) AS month,
       i.customer_id, i.country, l.stock_code, l.quantity, l.unit_price,
       l.quantity * l.unit_price AS amount
FROM invoice_lines AS l
JOIN invoices AS i ON i.invoice = l.invoice
JOIN products AS p ON p.stock_code = l.stock_code
WHERE p.is_goods = 1
  AND l.unit_price > 0
  AND i.kind IN ('sale', 'return');
""")
print("Created the view goods_lines.")

Run the next cell. It puts every line of `invoice_lines` into one of five parts, with its amount.

In [ ]:
sql("""
SELECT CASE WHEN i.kind = 'adjustment' THEN '5 bad-debt adjustment'
            WHEN p.is_goods = 0        THEN '4 postage, fees, manual'
            WHEN l.unit_price <= 0     THEN '3 goods at price 0'
            WHEN i.kind = 'return'     THEN '2 goods returned'
            ELSE                            '1 goods sold' END AS part,
       COUNT(*) AS lines,
       ROUND(SUM(l.quantity * l.unit_price), 2) AS amount
FROM invoice_lines AS l
JOIN invoices AS i ON i.invoice = l.invoice
JOIN products AS p ON p.stock_code = l.stock_code
GROUP BY part
ORDER BY part
""")

> **Check.** You should see £19,700,954.46 of goods sold and −£719,692.94 of goods returned.

Run the next cell. It shows the three totals that Kwame's reports used.

In [ ]:
sql("""
SELECT 'every raw line' AS step, COUNT(*) AS lines,
       ROUND(SUM(quantity * price), 2) AS amount
FROM raw_lines
UNION ALL
SELECT 'one copy of each line', COUNT(*), ROUND(SUM(quantity * unit_price), 2)
FROM invoice_lines
UNION ALL
SELECT 'net sales of goods', COUNT(*), ROUND(SUM(amount), 2)
FROM goods_lines
""")

> **Check.** You should see £19,287,250.57, £18,909,762.12 and £18,981,261.52.

**What this tells us.** The first total counts the nine copied days twice. The second total mixes goods with postage, fees and bad debt. The third is **net sales of goods**: sales minus returns, £18,981,261.52. The parts of the previous cell add up to the second total, so every pound is explained.

### Failure case: a total that grows when you join

Kwame wants net sales and the number of lines per country. He has a table of invoice totals and joins it to the lines. First, look at one invoice.

> **Predict.** Invoice `489434` has a total of £505.30 and 8 lines. What does `SUM(total)` give after the join?

In [ ]:
sql("""
WITH invoice_totals AS (
    SELECT invoice, ROUND(SUM(amount), 2) AS total
    FROM goods_lines
    WHERE invoice = '489434'
    GROUP BY invoice
)
SELECT t.invoice, t.total, g.line_no, g.stock_code, ROUND(g.amount, 2) AS amount
FROM invoice_totals AS t
JOIN goods_lines AS g ON g.invoice = t.invoice
""")

**What this tells us.** The join repeats the invoice's total on each of its 8 lines. `SUM(total)` would give 8 × £505.30 = £4,042.40. Now run Kwame's query on all invoices.

In [ ]:
sql("""
WITH invoice_totals AS (
    SELECT invoice, country, SUM(amount) AS total
    FROM goods_lines
    GROUP BY invoice, country
)
SELECT t.country,
       ROUND(SUM(t.total))  AS net_sales,
       COUNT(g.line_no)     AS lines
FROM invoice_totals AS t
JOIN goods_lines AS g ON g.invoice = t.invoice
GROUP BY t.country
ORDER BY net_sales DESC
LIMIT 3
""")

> **Check.** You should see more than £1,000,000,000 for the United Kingdom. The shop's whole net sales are £19 million.

The fix: sum the line amounts, at the grain of the lines. Run the next cell.

In [ ]:
sql("""
SELECT country,
       ROUND(SUM(amount)) AS net_sales,
       COUNT(*)           AS lines
FROM goods_lines
GROUP BY country
ORDER BY net_sales DESC
LIMIT 3
""")

> **Check.** You should see £16,221,081 for the United Kingdom.

**What this tells us.** The wrong total is about 63 times too large, because a UK invoice has many lines. No error appeared. The protection is a reconciliation: every total must add up to the checked net sales.

## 6. Monthly sales and the Christmas peak

Run the next cell. It computes the orders, sales, returns and net sales of each month.

In [ ]:
sql("""
SELECT month,
       COUNT(DISTINCT CASE WHEN kind = 'sale' THEN invoice END) AS orders,
       ROUND(SUM(CASE WHEN kind = 'sale'   THEN amount ELSE 0 END)) AS sales,
       ROUND(-SUM(CASE WHEN kind = 'return' THEN amount ELSE 0 END)) AS returns,
       ROUND(SUM(amount)) AS net_sales
FROM goods_lines
GROUP BY month
ORDER BY month
""", limit=None)

> **Check.** You should see 25 months. November 2011 has the most orders (2,751).

The next cell draws net sales by month.

In [ ]:
monthly = pd.read_sql("SELECT month, net_sales FROM (SELECT month, SUM(amount) AS net_sales FROM goods_lines GROUP BY month) ORDER BY month", con)
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed here; the lesson page has the chart.")
else:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(monthly["month"], monthly["net_sales"] / 1000, marker="o")
    ax.set_ylabel("net sales of goods, £ thousands")
    ax.tick_params(axis="x", rotation=90)
    plt.show()

**What this tells us.** Sales climb from September and peak in **November**, not December. The customers are shops: they buy before their own Christmas season.

Run the next cell. `LAG(net_sales, 12)` takes the value of 12 rows before. The query also shows `LAG(month, 12)`, so you can see that 12 rows back is really the same month last year.

In [ ]:
sql("""
WITH monthly AS (
    SELECT month, SUM(amount) AS net_sales
    FROM goods_lines
    GROUP BY month
),
with_last_year AS (
    SELECT month, net_sales,
           LAG(month, 12)     OVER (ORDER BY month) AS month_12_rows_back,
           LAG(net_sales, 12) OVER (ORDER BY month) AS net_sales_last_year
    FROM monthly
)
SELECT month, ROUND(net_sales) AS net_sales, month_12_rows_back,
       ROUND(net_sales_last_year) AS net_sales_last_year,
       ROUND(100.0 * (net_sales - net_sales_last_year) / net_sales_last_year, 1) AS change_pct
FROM with_last_year
WHERE month >= '2010-12'
""", limit=None)

> **Check.** You should see `2011-11  1432735.0  2010-11  1400559.0  2.3` and −42.0% for December 2011.

**What this tells us.** November 2011 was 2.3% above November 2010. December 2011 is not a fall of 42%: the data stops on 9 December. Run the next cell to compare the same nine days in each year.

In [ ]:
sql("""
SELECT substr(invoiced_at, 1, 4) AS year,
       ROUND(SUM(amount)) AS net_sales_1_to_9_dec,
       COUNT(DISTINCT CASE WHEN kind = 'sale' THEN invoice END) AS orders
FROM goods_lines
WHERE substr(invoiced_at, 6, 5) BETWEEN '12-01' AND '12-09'
GROUP BY year
""")

In [ ]:
sql("""
SELECT CASE WHEN month <= '2010-11' THEN 'Dec 2009 to Nov 2010'
            ELSE 'Dec 2010 to Nov 2011' END AS trading_year,
       ROUND(SUM(amount)) AS net_sales,
       ROUND(SUM(CASE WHEN substr(month, 6, 2) IN ('09', '10', '11') THEN amount END)) AS sep_to_nov,
       ROUND(100.0 * SUM(CASE WHEN substr(month, 6, 2) IN ('09', '10', '11') THEN amount END)
             / SUM(amount), 1) AS sep_to_nov_pct
FROM goods_lines
WHERE month <= '2011-11'
GROUP BY trading_year
""")

> **Check.** You should see £441,367 for 1 to 9 December 2011, and September to November at 37.5% of the trading year to November 2011.

**What this tells us.** The first nine days of December 2011 were 9.7% above the same days of 2010. September to November brings more than a third of a year's net sales. For Ines, the season starts in September, so the stock must arrive by late August.

> **Your turn.** Which five countries had the most net sales from September to November 2011? Return `country` and `net_sales` (rounded to whole pounds with `ROUND()`), largest first, five rows.

In [ ]:
country_q4 = """
SELECT country
FROM goods_lines
-- Add the months, the groups, the order and the limit
"""
sql(country_q4)

In [ ]:
check(country_q4, rows=5, columns=2, expected="b617f60c80df87c1", ordered=True)

## 7. Returns: a rate, and the orders behind the spikes

Run the next cell. For each month it shows the returns, the return rate (returns ÷ sales) and the largest single return line, found with `ROW_NUMBER()`.

In [ ]:
sql("""
WITH return_lines AS (
    SELECT month, line_no, invoice, customer_id, stock_code, quantity,
           -amount AS value,
           ROW_NUMBER() OVER (PARTITION BY month ORDER BY -amount DESC, line_no) AS place
    FROM goods_lines
    WHERE kind = 'return'
),
monthly AS (
    SELECT month,
           SUM(CASE WHEN kind = 'sale' THEN amount ELSE 0 END) AS sales,
           -SUM(CASE WHEN kind = 'return' THEN amount ELSE 0 END) AS returns
    FROM goods_lines
    GROUP BY month
)
SELECT m.month, ROUND(m.returns) AS returns,
       ROUND(100.0 * m.returns / m.sales, 1) AS return_rate_pct,
       ROUND(r.value) AS largest_line,
       ROUND(100.0 * r.value / m.returns) AS largest_line_pct
FROM monthly AS m
JOIN return_lines AS r ON r.month = m.month AND r.place = 1
ORDER BY m.month
""", limit=None)

> **Check.** You should see 13.6% in January 2011 and 28.3% in December 2011. In those months, one line is 84% and 97% of the returns.

**What this tells us.** In most months, 1% to 4% of sales come back. Two months jump, and one line explains each jump. Run the next cell. It finds the order of each of those two return lines.

In [ ]:
sql("""
SELECT r.invoice AS return_invoice, s.invoice AS sale_invoice, r.customer_id,
       r.stock_code, s.quantity, s.invoiced_at AS ordered_at, r.invoiced_at AS returned_at,
       ROUND(s.amount) AS value
FROM goods_lines AS r
JOIN goods_lines AS s
  ON  s.customer_id = r.customer_id
  AND s.stock_code  = r.stock_code
  AND s.quantity    = -r.quantity
  AND s.kind = 'sale'
  AND s.invoiced_at <= r.invoiced_at
  AND r.invoiced_at <= datetime(s.invoiced_at, '+1 hour')
WHERE r.kind = 'return' AND -r.amount > 10000
""")

In [ ]:
sql("""
WITH monthly AS (
    SELECT month,
           SUM(CASE WHEN kind = 'sale' THEN amount ELSE 0 END) AS sales,
           -SUM(CASE WHEN kind = 'return' THEN amount ELSE 0 END) AS returns,
           SUM(CASE WHEN invoice IN ('541431', '581483', 'C541433', 'C581484')
                    THEN ABS(amount) ELSE 0 END) / 2 AS mistaken
    FROM goods_lines
    GROUP BY month
)
SELECT month,
       ROUND(100.0 * returns / sales, 1) AS return_rate_pct,
       ROUND(100.0 * (returns - mistaken) / (sales - mistaken), 1) AS without_mistaken_orders_pct
FROM monthly
WHERE month IN ('2011-01', '2011-12')
""")

> **Check.** You should see two orders cancelled 16 and 12 minutes after they were placed, and return rates of 2.4% and 1.3% without them.

**What this tells us.** The spikes are two mistaken orders, each cancelled within minutes. A stricter returns policy would not change them. Kwame should report the return rate with and without such orders, and look at the usual months.

## 8. Do new customers come back?

A **cohort** is a group of customers who started in the same month. Run the next cell. Its CTE steps find each customer's first month, then check if they bought again within the next three months.

In [ ]:
sql("""
WITH active AS (          -- one row per customer and month with a purchase
    SELECT DISTINCT customer_id, month
    FROM goods_lines
    WHERE kind = 'sale' AND customer_id IS NOT NULL
),
first_month AS (          -- the month each customer first appears
    SELECT customer_id, MIN(month) AS cohort
    FROM active
    GROUP BY customer_id
),
came_back AS (            -- did they buy again in the next three months?
    SELECT f.cohort, f.customer_id,
           MAX(a.month >  f.cohort AND
               a.month <= strftime('%Y-%m', f.cohort || '-01', '+3 months')) AS back
    FROM first_month AS f
    JOIN active AS a ON a.customer_id = f.customer_id
    GROUP BY f.cohort, f.customer_id
)
SELECT cohort, COUNT(*) AS customers,
       ROUND(100.0 * AVG(back), 1) AS back_within_3_months_pct
FROM came_back
WHERE cohort <= '2011-08'
GROUP BY cohort
ORDER BY cohort
""", limit=None)

> **Check.** You should see 951 customers in the December 2009 cohort, and 63.0% of them came back within three months.

**What this tells us.** The first cohort looks far better than the others, and it is also the largest. That is suspicious. The data starts in December 2009, but the shop is older. So "first month in the data" is not "first purchase" for customers who bought before.

Run the next cell. It counts return lines whose product the customer never bought earlier **in the data**.

In [ ]:
sql("""
WITH return_lines AS (
    SELECT r.month, r.line_no,
           EXISTS (SELECT 1 FROM goods_lines AS s
                   WHERE s.customer_id = r.customer_id
                     AND s.stock_code = r.stock_code
                     AND s.kind = 'sale'
                     AND s.invoiced_at <= r.invoiced_at) AS sale_in_data
    FROM goods_lines AS r
    WHERE r.kind = 'return' AND r.customer_id IS NOT NULL
)
SELECT CASE WHEN month <= '2010-02' THEN '1: Dec 2009 to Feb 2010'
            WHEN month <= '2010-11' THEN '2: Mar to Nov 2010'
            ELSE                         '3: Dec 2010 to Dec 2011' END AS period,
       COUNT(*) AS return_lines,
       ROUND(100.0 * SUM(NOT sale_in_data) / COUNT(*), 1) AS sold_before_the_data_pct
FROM return_lines
GROUP BY period
ORDER BY period
""")

In [ ]:
sql("""
WITH active AS (
    SELECT DISTINCT customer_id, month
    FROM goods_lines
    WHERE kind = 'sale' AND customer_id IS NOT NULL
),
first_month AS (
    SELECT customer_id, MIN(month) AS cohort FROM active GROUP BY customer_id
),
came_back AS (
    SELECT f.cohort, f.customer_id,
           MAX(a.month >  f.cohort AND
               a.month <= strftime('%Y-%m', f.cohort || '-01', '+3 months')) AS back
    FROM first_month AS f
    JOIN active AS a ON a.customer_id = f.customer_id
    GROUP BY f.cohort, f.customer_id
)
SELECT CASE WHEN cohort = '2009-12' THEN 'Dec 2009: first month of the data'
            WHEN cohort = '2010-12' THEN 'Dec 2010: 12 months of history before' END AS cohort,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(back), 1) AS back_within_3_months_pct
FROM came_back
WHERE cohort IN ('2009-12', '2010-12')
GROUP BY cohort
""")

> **Check.** You should see 42.8% for December 2009 to February 2010, and 1.7% from December 2010.

**What this tells us.** In the first three months, 42.8% of returns are of goods bought before the data starts: proof of an older history. The December 2009 cohort is mostly old customers. A December cohort with a year of history before it (December 2010) came back at 18.4%, not 63.0%. The dashboard's "falling retention" is mostly this trap.

> **Your turn: change one thing.** The cell below is the cohort query with three months. Change it to **six** months: the window, the column name `back_within_6_months_pct`, and the last cohort (`2011-05`, because the data must cover six months after it). Run it, then run the check.

In [ ]:
cohorts_6 = """
WITH active AS (
    SELECT DISTINCT customer_id, month
    FROM goods_lines
    WHERE kind = 'sale' AND customer_id IS NOT NULL
),
first_month AS (
    SELECT customer_id, MIN(month) AS cohort FROM active GROUP BY customer_id
),
came_back AS (
    SELECT f.cohort, f.customer_id,
           MAX(a.month >  f.cohort AND
               a.month <= strftime('%Y-%m', f.cohort || '-01', '+3 months')) AS back
    FROM first_month AS f
    JOIN active AS a ON a.customer_id = f.customer_id
    GROUP BY f.cohort, f.customer_id
)
SELECT cohort, COUNT(*) AS customers,
       ROUND(100.0 * AVG(back), 1) AS back_within_3_months_pct
FROM came_back
WHERE cohort <= '2011-08'
GROUP BY cohort
ORDER BY cohort
"""
sql(cohorts_6)

In [ ]:
check(cohorts_6, rows=18, columns=3, expected="38ef15e120c250de", ordered=True)

## 9. The top product in each country, with ties

Ines wants the product that most orders contained in each country, from September to November 2011. Run the next cell. It shows `ROW_NUMBER()` and `RANK()` side by side for two countries.

In [ ]:
sql("""
WITH orders_per_product AS (
    SELECT country, stock_code, COUNT(DISTINCT invoice) AS orders
    FROM goods_lines
    WHERE kind = 'sale' AND month BETWEEN '2011-09' AND '2011-11'
    GROUP BY country, stock_code
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY country ORDER BY orders DESC) AS row_number,
           RANK()       OVER (PARTITION BY country ORDER BY orders DESC) AS rank
    FROM orders_per_product
)
SELECT country, stock_code, orders, row_number, rank
FROM ranked
WHERE country IN ('Spain', 'Switzerland') AND rank = 1
ORDER BY country, row_number
""")

In [ ]:
sql("""
WITH orders_per_product AS (
    SELECT country, stock_code, COUNT(DISTINCT invoice) AS orders
    FROM goods_lines
    WHERE kind = 'sale' AND month BETWEEN '2011-09' AND '2011-11'
    GROUP BY country, stock_code
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY country ORDER BY orders DESC) AS rank
    FROM orders_per_product
)
SELECT COUNT(DISTINCT country) AS countries,
       COUNT(DISTINCT CASE WHEN tied > 1 THEN country END) AS countries_with_a_tie_at_the_top
FROM (SELECT country, COUNT(*) AS tied FROM ranked WHERE rank = 1 GROUP BY country)
""")

> **Check.** You should see two products with `rank` 1 in Spain and three in Switzerland, and 16 of 30 countries with a tie at the top.

**What this tells us.** `ROW_NUMBER()` gives one winner even when products tie, and the choice can change between runs. `RANK()` shows the tie. So break ties with a rule that you can explain, and show how many products tied. Run the next cell.

In [ ]:
sql("""
WITH orders_per_product AS (
    SELECT country, stock_code, COUNT(DISTINCT invoice) AS orders,
           ROUND(SUM(amount)) AS net_sales
    FROM goods_lines
    WHERE kind = 'sale' AND month BETWEEN '2011-09' AND '2011-11'
    GROUP BY country, stock_code
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY country
                              ORDER BY orders DESC, net_sales DESC, stock_code) AS place,
           COUNT(*) OVER (PARTITION BY country, orders) AS same_orders
    FROM orders_per_product
)
SELECT r.country, r.stock_code, p.description, r.orders, r.net_sales,
       r.same_orders AS products_with_as_many_orders
FROM ranked AS r
JOIN products AS p ON p.stock_code = r.stock_code
WHERE r.place = 1 AND r.orders >= 6
ORDER BY r.orders DESC
""")

**What this tells us.** In the United Kingdom, a Christmas paper chain kit was in 758 orders. In the other countries shown, "the top product" is in 6 to 40 orders, and in the 18 countries that the query does not show, in 1 to 4 orders. That is too few to plan stock by country: Ines should plan small markets together.

## 10. Deliver a checked monthly table

Run the next cell. It builds `monthly_sales`, one row per month, with a flag for the incomplete last month.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS monthly_sales;
CREATE TABLE monthly_sales AS
WITH monthly AS (
    SELECT month,
           COUNT(DISTINCT substr(invoiced_at, 1, 10)) AS trading_days,
           COUNT(DISTINCT CASE WHEN kind = 'sale' THEN invoice END) AS orders,
           COUNT(DISTINCT CASE WHEN kind = 'sale' THEN customer_id END) AS customers,
           ROUND(SUM(CASE WHEN kind = 'sale' THEN amount ELSE 0 END), 2) AS sales,
           ROUND(-SUM(CASE WHEN kind = 'return' THEN amount ELSE 0 END), 2) AS returns,
           ROUND(SUM(amount), 2) AS net_sales
    FROM goods_lines
    GROUP BY month
)
SELECT month, trading_days, orders, customers, sales, returns, net_sales,
       ROUND(returns / sales, 4) AS return_rate,
       LAG(net_sales, 12) OVER (ORDER BY month) AS net_sales_last_year,
       ROUND(net_sales / LAG(net_sales, 12) OVER (ORDER BY month) - 1, 4) AS change_vs_last_year,
       month < '2011-12' AS complete_month
FROM monthly
ORDER BY month;
""")
sql("SELECT * FROM monthly_sales WHERE month >= '2011-09'")

Run the next cell. Each rule is a query that must return no rows, as in [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/).

> **Check.** You should see `6 of 6 rules pass`.

In [ ]:
rules = {
    "one row per month":
        "SELECT month FROM monthly_sales GROUP BY month HAVING COUNT(*) > 1",
    "no missing month":
        """SELECT month FROM (SELECT month, LAG(month) OVER (ORDER BY month) AS previous
                              FROM monthly_sales)
           WHERE previous IS NOT NULL
             AND month <> strftime('%Y-%m', previous || '-01', '+1 month')""",
    "net sales = sales - returns":
        "SELECT month FROM monthly_sales WHERE ABS(sales - returns - net_sales) > 0.02",
    "total equals goods_lines":
        """SELECT 1 FROM (SELECT SUM(net_sales) AS t FROM monthly_sales)
           WHERE ABS(t - (SELECT SUM(amount) FROM goods_lines)) > 0.5""",
    "return rate between 0 and 0.3":
        "SELECT month FROM monthly_sales WHERE return_rate < 0 OR return_rate > 0.3",
    "only the last month is incomplete":
        """SELECT month FROM monthly_sales
           WHERE complete_month = (month = (SELECT MAX(month) FROM monthly_sales))""",
}
failed = 0
for name, rule in rules.items():
    problems = con.execute(rule).fetchall()
    failed += bool(problems)
    print(f"{'FAIL' if problems else 'pass'}  {name}" + (f"  {problems[:3]}" if problems else ""))
print(f"{len(rules) - failed} of {len(rules)} rules pass")

Run the next cell. It writes `monthly_sales.csv` and `LINEAGE.md`: where the numbers come from, what each step removed, and the checksum of the file.

In [ ]:
import hashlib

monthly = pd.read_sql("SELECT * FROM monthly_sales ORDER BY month", con)
monthly.to_csv("monthly_sales.csv", index=False)
digest = hashlib.sha256(Path("monthly_sales.csv").read_bytes()).hexdigest()
counts = {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
          for t in ["raw_lines", "invoice_lines", "goods_lines", "monthly_sales"]}
Path("LINEAGE.md").write_text(f"""# monthly_sales.csv: lineage

- Source: Online Retail II, Daqing Chen, UCI Machine Learning Repository,
  https://doi.org/10.24432/C5CG6D, CC BY 4.0. File online_retail_ii.csv.gz
  (SHA-256 {DATA_SHA256}), the two Excel sheets joined in order.
- raw_lines: {counts["raw_lines"]:,} lines, loaded unchanged, with line_no = position in the file.
- invoice_lines: {counts["invoice_lines"]:,} lines. Removed the 22,523 lines of sheet 2 dated
  1-9 Dec 2010 (line_no >= 525462): an exact copy of sheet 1. Kept 11,812 repeats inside one sheet.
- goods_lines: {counts["goods_lines"]:,} lines. Goods only (no postage, fees, manual lines,
  vouchers or bad-debt adjustments), unit price above 0, sales and returns.
- monthly_sales: {counts["monthly_sales"]} months, Dec 2009 to Dec 2011. Net sales = sales - returns, in GBP.
  Dec 2011 has 9 days only (complete_month = 0). Guest orders (no customer ID) count in sales,
  not in customers.
- Checks: the six validation rules in the notebook pass; the total equals goods_lines.
- Output: monthly_sales.csv, SHA-256 {digest}
""", encoding="utf-8")
print(f"monthly_sales.csv: {len(monthly)} rows")
print()
print(Path("LINEAGE.md").read_text(encoding="utf-8"))

> **Check.** You should see `monthly_sales.csv: 25 rows`.

Run the next cell to see the chart of the delivered table.

In [ ]:
monthly = pd.read_sql("SELECT month, net_sales FROM monthly_sales ORDER BY month", con)
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed here; the lesson page has the chart.")
else:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(monthly["month"], monthly["net_sales"] / 1000, marker="o")
    ax.set_ylabel("net sales of goods, £ thousands")
    ax.tick_params(axis="x", rotation=90)
    plt.show()

## 11. Recommendations, limits and your portfolio

The [lesson page](https://learning.nextia-ai.com/courses/sql/m07/explain-an-online-shops-sales/#recommendations) has the recommendations for Ines and Kwame, the limits, and a guide to turn this notebook into a portfolio project: a README, this notebook, `monthly_sales.csv` and `LINEAGE.md`. The two files are in the folder `explain-an-online-shops-sales`. Do not publish `shop.db` or the data file: link to the source.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m07/explain-an-online-shops-sales/#knowledge-check).

In this case study you found a copied sheet with one `LAG`, built four tables with keys and checked them, and reconciled three revenue totals to £18,981,261.52 of net sales of goods. You saw a join make a total 63 times too large. You showed that the Christmas season starts in September, that two return spikes were two mistaken orders, and that the best cohort was only the start of the data. Next: a leak-free table for city repair requests.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Prepare a leak-free table for city repair requests](https://learning.nextia-ai.com/courses/sql/m07/prepare-a-leak-free-table-for-city-repair-requests/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m07/explain-an-online-shops-sales/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real (UCI Online Retail II, CC BY 4.0); the people in the story are made up.